<a href="https://colab.research.google.com/github/AnnaBastin/Company-WikiBot/blob/main/company.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#this install necessary requirements only
#also please ensure that your google api key has been added to secret keys, in google colab. the key must have billing as well.
#your own key will be read and used while running this code.
!pip install -q -U langchain langchain-community langchain-text-splitters chromadb
!pip install -q google-generativeai langchain-google-genai
!pip install -q pypdf python-docx docx2txt
!pip install -U langchain-google-genai google-generativeai
#!pip install python-dotenv

In [2]:
#this will allow effective cloning of my repo into your copy of my notebook
#therefore all text files from my github can be read
!git clone https://github.com/AnnaBastin/Company-WikiBot.git

fatal: destination path 'Company-WikiBot' already exists and is not an empty directory.


In [3]:
#NOT NECESSARY
#this is just to see if cloning is successful, output should be Company-WikiBot  sample_data
!ls

chroma_db  Company-WikiBot  documents  sample_data


In [4]:
#NOT NECESSARY
#this is just to see if Company-Wikibot is containing the files needed, output should be company.ipynb  documents  README.md
!ls Company-WikiBot


company.ipynb  documents  README.md


In [5]:
#NOT NECESSARY
 #run only for checking if documents contains the names of any documents, this checks if docs are being returned
!ls Company-Wiki/documents

ls: cannot access 'Company-Wiki/documents': No such file or directory


In [6]:
import os
import glob

from google.colab import files

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader
)

from langchain_community.vectorstores import Chroma

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [7]:
from google.colab import userdata
import google.generativeai as genai

api_key = userdata.get("GOOGLE_API_KEY")

genai.configure(api_key=api_key)

print("API configured successfully")

API configured successfully


In [8]:
DOCUMENTS_PATH = "/content/Company-WikiBot/documents"
documents = []

all_files = glob.glob(f"{DOCUMENTS_PATH}/**/*", recursive=True)

print("Files found:")
for f in all_files:
    print(f)

for file_path in all_files:

    if os.path.isdir(file_path):
        continue

    try:
        if file_path.endswith(".txt"):
            loader = TextLoader(file_path)

        elif file_path.endswith(".pdf"):
            loader = PyPDFLoader(file_path)

        elif file_path.endswith(".docx"):
            loader = Docx2txtLoader(file_path)

        else:
            print("Skipping:", file_path)
            continue

        documents.extend(loader.load())
        print("Loaded:", file_path)

    except Exception as e:
        print("Error:", file_path, e)

print("Total docs:", len(documents))

Files found:
/content/Company-WikiBot/documents/employee handbook.txt
/content/Company-WikiBot/documents/employee of conflict of interest.txt
/content/Company-WikiBot/documents/open door policy.txt
/content/Company-WikiBot/documents/bloggin and social media policy.txt
/content/Company-WikiBot/documents/work from home policy.txt
Loaded: /content/Company-WikiBot/documents/employee handbook.txt
Loaded: /content/Company-WikiBot/documents/employee of conflict of interest.txt
Loaded: /content/Company-WikiBot/documents/open door policy.txt
Loaded: /content/Company-WikiBot/documents/bloggin and social media policy.txt
Loaded: /content/Company-WikiBot/documents/work from home policy.txt
Total docs: 5


In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 168


In [10]:
#sometimes some model versions will not be available
#this code snippet is only to check which versions can be supported for you
#not really necessary to be run
"""import google.generativeai as genai
import os

#genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
#genai.configure(api_key=api_key)

models = genai.list_models()

for m in models:
    print("MODEL:", m.name)
    print("SUPPORTED METHODS:", m.supported_generation_methods)
    print("-" * 50)

for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print("EMBEDDING MODEL:", m.name)"""

'import google.generativeai as genai\nimport os\n\n#genai.configure(api_key=os.environ["GOOGLE_API_KEY"])\n#genai.configure(api_key=api_key)\n\nmodels = genai.list_models()\n\nfor m in models:\n    print("MODEL:", m.name)\n    print("SUPPORTED METHODS:", m.supported_generation_methods)\n    print("-" * 50)\n\nfor m in genai.list_models():\n    if "embedContent" in m.supported_generation_methods:\n        print("EMBEDDING MODEL:", m.name)'

In [11]:
#reads your API key from google colab secrets
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [12]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

test = embeddings.embed_query("hello world")
print("Embedding length:", len(test))



Embedding length: 3072


In [13]:
"""vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="/content/chroma_db"
)

retriever = vectorstore.as_retriever()"""

import time
from langchain_community.vectorstores import Chroma

BATCH_SIZE = 15          # increased (was 5)
SLEEP_BETWEEN_BATCHES = 2  # reduced (was 8)

def build_vectorstore(chunks):
    vectorstore = None

    for i in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[i:i + BATCH_SIZE]

        # single retry with light backoff (not exponential overkill)
        for attempt in range(3):
            try:
                if vectorstore is None:
                    vectorstore = Chroma.from_documents(
                        documents=batch,
                        embedding=embeddings,
                        persist_directory="/content/chroma_db"
                    )
                else:
                    vectorstore.add_documents(batch)

                break  # success

            except Exception as e:
                wait = 2 * (attempt + 1)  # 2s, 4s, 6s
                time.sleep(wait)

        time.sleep(SLEEP_BETWEEN_BATCHES)

    return vectorstore


vectorstore = build_vectorstore(chunks)
retriever = vectorstore.as_retriever()

In [14]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

In [15]:
query = "what is the policy on maternity "

retrieved_docs = retriever.invoke(query)

print("Retrieved docs:", len(retrieved_docs))

if len(retrieved_docs) == 0:
    print("No relevant documents found.")
else:
    context = "\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
    Answer ONLY using the context below.

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.invoke(prompt)
    clean_text = response.content[0]["text"]
    print(clean_text)
    #print(output)

Retrieved docs: 4
Based on the provided context, the policy on maternity is as follows:

* **Paid Leave:** Employees are eligible for 2 weeks of paid maternity leave, which can start before or after the due date.
* **Complications:** In the case of complications, 1 week of unpaid leave will be granted.
* **Arrangements:** Employees must reach out to HR as soon as possible to make arrangements.
* **Notice Period:** If you are expecting a child through childbirth, you must inform your manager at least [2 months] in advance.
